# Day two — Biodiversity, Land Use and Habitat Change Detection
## *PART I: Philippines Climate Zones, revisited*

Yesterday, we wrapped up with the figures from Beck et al. (2023) that showed us both the current Köppen-Geiger zones and those projected under a future climate change scenario. First things first, re-run the setup cell below if the project is not loaded in your Google Colab environment yet:

In [ ]:
# 1
import os                                                                          # in English:
if not os.path.exists("/content/USLS-Workshop"):                                   # IF workshop isn't downloaded yet:
    !git clone "https://github.com/Cas-Dec/USLS-Workshop.git" /content/USLS-Workshop  # download the workshop
# open the workshop
%cd /content/USLS-Workshop
!pip install -q .                                                                  # install what is needed to do the workshop

#from google.colab import drive
#drive.mount('/content/drive')                                                      # connect to your Google Drive

In [ ]:
# 2
import numpy as np                                                      # anything with numbers
import pandas as pd                                                     # anything with tables
import xarray as xr                                                     # geospatial data
import matplotlib.pyplot as plt                                         # make visuals
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from workshop_utils import PROCESSED_DIR                                # PROCESSED_DIR is the data/processed/ directory, where the data we need is stored

print("Ready!")

### Day two and we're expert programmers: let's build our own climate zones!

Yesterday you classified **Bacolod alone** by hand-computing four numbers and running them through an `if/elif` chain. Today we generalize that into a real, reusable *function*, finish the decision tree we left incomplete, and run it not just for one city but across the **entire Philippines**, at **three points in time**.

Three files already sit in `PROCESSED_DIR` (monthly `air_temperature` + `precipitation`, 0.1° grid, Philippines-wide):
- Past: `climate_philippines_1901_1930_mean.nc`
- Present: `climate_philippines_1991_2020_mean.nc`
- Future (SSP4-6.0): `climate_philippines_2071_2099_ssp460_mean.nc`

*(We also asked yesterday: could we get our hands on a land-sea mask to go with this? Turns out we don't need to download one — the ocean is simply `NaN` in these files, since Beck et al.'s land climate ensemble has no value over open water. That NaN pattern **is** our land-sea mask, for free.)*

See the outline below for the exercise-by-exercise plan — nothing to run yet, this is for review before we write the actual fill-in-the-blank cells.

### New tools for today: `def` functions and `for` loops

Yesterday you wrote a decision tree once, for one city, by hand. Today we want to run the *same* decision tree for **thousands of grid points**, three different points in time. Copy-pasting the same 20 lines thousands of times isn't going to work — we need two new tools:

- **`def` functions** — package a block of code into something you can *name* and *reuse*, with different inputs each time. 👉 https://docs.python.org/3/tutorial/controlflow.html#defining-functions
- **`for` loops** — repeat a block of code automatically, once per item in a list (or grid cell). 👉 https://docs.python.org/3/tutorial/controlflow.html#for-statements

Let's start with functions.

In [ ]:
# =============================================================
# 💡 EXAMPLE — a simple function
# =============================================================

def celsius_to_fahrenheit(temp_c):        # 💡 "def" starts a function; temp_c is its input (a "parameter")
    temp_f = temp_c * 9 / 5 + 32
    return temp_f                         # 💡 "return" sends the result back out

# Now we can call it as many times as we want, with different inputs:
print(celsius_to_fahrenheit(0))
print(celsius_to_fahrenheit(28))
print(celsius_to_fahrenheit(100))

🖊️ **Try for yourself!** Make a simple function that tells you how long your name is:

In [ ]:
def how_long_name(name):
    return len(name)                      # ✏️ remember: "len(...)" counts the number of characters in a string

print(how_long_name("Cas"))         # ✏️ try out some names

A function can take several inputs, and a `return` statement always ends the function immediately. Notice `temp_c` only exists *inside* the function — that's its own little workspace.

---

### Step 1 — wrap yesterday's decision tree into a function

Let's reload Bacolod's climatology from yesterday (same files, same numbers) so we have something to test our function on:

In [ ]:
# --- Reload Bacolod's 1991-2020 monthly climatology (same as yesterday) ---
t2m_monthly = xr.open_dataarray(PROCESSED_DIR / "t2m_bacolod_monthly.nc")
tp_monthly  = xr.open_dataarray(PROCESSED_DIR / "tp_bacolod_monthly.nc")

coldest_month_T = float(t2m_monthly.min())
warmest_month_T = float(t2m_monthly.max())
annual_mean_T   = float(t2m_monthly.mean())
annual_precip   = float(tp_monthly.sum())
driest_month_P  = float(tp_monthly.min())

print(f"Coldest month mean temperature : {coldest_month_T:.1f} °C")
print(f"Warmest month mean temperature : {warmest_month_T:.1f} °C")
print(f"Annual mean temperature        : {annual_mean_T:.1f} °C")
print(f"Annual precipitation           : {annual_precip:.0f} mm")
print(f"Driest month precipitation     : {driest_month_P:.0f} mm")

### 🖊️ Your turn: wrap the Group A logic into a function

Below is yesterday's logic for detecting **Group A (tropical)**, now with all three subtypes filled in — including **Am (monsoon)**, using the formula from yesterday's `if/elif/else` example (`driest < 60 and annual >= 25 * (100 - driest)`). Everything that *isn't* Group A is temporarily lumped into `"other"` (we'll fix Groups B/C/D/E in Step 2). Your job is just the *wrapping*: turn it into a function called `classify_koppen` that takes the five numbers we just computed and `return`s the resulting climate code.

**Tips:**
- The function needs 5 parameters, in this order: `coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P`
- End the function with `return group + subtype`

In [ ]:
# ✏️ Call the function "classify_koppen":
def classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P):

    # Step 1: main group (only Group A is implemented for now)
    if coldest_month_T >= 18:
        group = "A"
    else:
        group = "other"          # ⚠️ Groups B/C/D/E come in Step 2

    # Step 2: subtype (only defined for Group A)
    if group == "A":
        if driest_month_P >= 60:
            subtype = "f"                                                     # Af — rainforest
        elif driest_month_P < 60 and annual_precip >= 25 * (100 - driest_month_P):
            subtype = "m"                                                     # Am — monsoon
        else:
            subtype = "w"                                                     # Aw — savanna
    else:
        subtype = ""

    return f"{group}{subtype}"   # ✏️ return climate code (group + subtype). hint: use the f-string formatting we learned yesterday: f"{...}{...}"


# --- Sanity check: does Bacolod still come out the same as yesterday? ---
bacolod_climate = classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P)
print(f"Bacolod's Köppen-Geiger climate type: {bacolod_climate}")

> 💬 **Discussion:** Does `bacolod_climate` match what you found by hand yesterday (Af)? A function is just a *reusable wrapper* — the science inside hasn't changed yet.

---

### Step 2 — completing the decision tree

Yesterday, Group B (arid) used a placeholder threshold, and Groups C, D, and E weren't implemented at all. Let's finish the job properly, using the actual criteria from [Peel, Finlayson & McMahon (2007)](https://doi.org/10.5194/hess-11-1633-2007) — the paper behind the modern, widely-used version of the Köppen-Geiger map (also what [Beck et al. 2023](https://doi.org/10.1038/s41597-023-02549-6) build on):

| Group | Criterion |
|---|---|
| **A** (Tropical) | coldest month ≥ 18°C |
| **B** (Arid) | annual precipitation below an *aridity threshold* (see below) |
| **C** (Temperate) | coldest month between −3°C and 18°C, warmest month ≥ 10°C |
| **D** (Continental) | coldest month < −3°C, warmest month ≥ 10°C |
| **E** (Polar) | warmest month < 10°C |

**The aridity threshold (Group B)** isn't a single fixed number — it depends on *how a place's rain is distributed across the year*, not just on how much rain falls in total. A place that gets all its rain in summer can "afford" to be drier overall than a place where rain spreads evenly, because summer rain evaporates faster. The real formula (using `annual_mean_T` in °C and `annual_precip` in mm):

- If **≥70%** of annual rain falls Apr–Sep ("summer" in the Northern Hemisphere): `threshold = 20 × annual_mean_T + 280`
- If **≥70%** of annual rain falls Oct–Mar ("winter"): `threshold = 20 × annual_mean_T`
- Otherwise (no strong season): `threshold = 20 × annual_mean_T + 140`
- A place is **Group B** if `annual_precip < threshold`

Since the Philippines sits entirely in the Northern Hemisphere, "summer" always means Apr–Sep here — no extra hemisphere logic needed. Let's see how to compute the % of rain falling Apr–Sep from a 12-month array:

In [ ]:
# =============================================================
# 💡 EXAMPLE — slicing a 12-month array to get Apr-Sep
# =============================================================
monthly_P = tp_monthly.values                # 12 numbers, Jan=index 0, ..., Dec=index 11

# ⚠️ Remember: Python uses zero-based indexing, so Jan=0, Feb=1, ..., Dec=11
summer_months = monthly_P[3:9]               # 💡 slicing: index 3 up to (not including) 9 -> Apr, May, Jun, Jul, Aug, Sep
print("Apr-Sep monthly precip:", summer_months)

summer_precip_pct = summer_months.sum() / monthly_P.sum() * 100
print(f"Percentage of annual rain falling Apr-Sep: {summer_precip_pct:.0f}%")

🖊️ **Try for yourself!** Slice the string below in different ways:

In [ ]:
some_string = "Hey look it's already day two of the workshop!"

first_letter = some_string[0]                 # ✏️ slice the string to get "H"
print(first_letter)

letter_three_to_five = some_string[2:5]     # ✏️ slice the string to get "y l"
print(letter_three_to_five)

# hint: use negative indexing to count from the end of the string
last_letter = some_string[-1]                  # ✏️ slice the string to get "!". 
print(last_letter)

day_two = some_string[22:29]                  # ✏️ slice the string to get "day two"
print(day_two)

### 🖊️ Your turn: finish the decision tree

Extend `classify_koppen` to properly determine Groups B, C, D and E. The function now needs a 6th parameter, `summer_precip_pct`, to compute the aridity threshold.

**Tips:**
- Work through the table above in this order: A, then B (arid), then E (polar), then D (continental), then C (temperate) — this order matters, because D and E both need `coldest_month_T < -3`/`warmest_month_T < 10` type checks, and E should "win" over D when both could apply.
- `p_threshold` should be computed *before* the group `if/elif` chain, since Group B's condition depends on it.

In [ ]:
# ✏️ Complete this decision tree:

def classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct):

    # Step 1: aridity threshold (needed for Group B)
    if summer_precip_pct >= 70:
        p_threshold = 20 * annual_mean_T + 280             # ✏️ 20 * annual_mean_T + 280
    elif summer_precip_pct <= 30:
        p_threshold = 20 * annual_mean_T             # ✏️ 20 * annual_mean_T
    else:
        p_threshold = 20 * annual_mean_T + 140             # ✏️ 20 * annual_mean_T + 140

    # Step 2: main group
    if coldest_month_T >= 18:
        group = "A"                          # Tropical
    elif annual_precip < p_threshold:                # ✏️ compare annual_precip against p_threshold
        group = "B"                        # ✏️ Arid
    elif warmest_month_T < 10:                           # ✏️ which variable decides Polar?
        group = "E"                        # ✏️ Polar
    elif coldest_month_T < -3:              # ✏️ Continental threshold (see table above)
        group = "D"                        # ✏️ Continental
    else:
        group = "C"                        # ✏️ Temperate

    # Step 3: subtype (only defined for Group A) — unchanged from Step 1
    if group == "A":
        if driest_month_P >= 60:
            subtype = "f"                                                     # Af — rainforest
        elif driest_month_P < 60 and annual_precip >= 25 * (100 - driest_month_P):
            subtype = "m"                                                     # Am — monsoon
        else:
            subtype = "w"                                                     # Aw — savanna
    else:
        subtype = ""

    return group + subtype


# --- Sanity check: recompute summer_precip_pct for Bacolod and re-classify ---
summer_precip_pct = tp_monthly.values[3:9].sum() / tp_monthly.values.sum() * 100
bacolod_climate = classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T,
                                   annual_precip, driest_month_P, summer_precip_pct)
print(f"Bacolod's Köppen-Geiger climate type: {bacolod_climate}")

> 💬 **Discussion:** Bacolod is very wet year-round, so it was never at risk of landing in Group B. Are there any parts of the Philippines *do* you expect to be dry enough for Group B?

---

### Step 3 — looping over a handful of cities

`classify_koppen` now works for any location — we just need to *call it repeatedly*. Before jumping to a full 170×135 grid, let's warm up with a `for` loop over a short, hand-written list of a few Philippine cities.

In [ ]:
# =============================================================
# 💡 EXAMPLE — a for loop
# =============================================================
fruits = ["mango", "banana", "durian"]

for fruit in fruits:                 # 💡 "fruit" takes each value in the list, one at a time
    print(f"I like {fruit}!")

💡 Easy, right? You can also loop over pairs of so-called "tuples", 🖊️ **fill in what's missing**:

In [ ]:
# 💡 A list of tuples (pairs):
prices = [("mango", 60), ("kiwi", 10), ("pineapple", 20)]        # ✏️ choose fuits and their prices

# 💡 You can loop over them as well:
for name, price in prices:                                     # ✏️ fill in what's missing
    print(f"{name} costs {price} pesos")

💡 Or use `zip` to combine lists into a list of tuples! Look:

In [ ]:
names = ["Alice", "Bob", "Charlie"]
ages = [25, 30, 35]                                      # ✏️ give Alice, Bob and Charlie some ages

# 💡 "zip" combines two lists into a list of tuples
for name, age in zip(names, ages):                              # ✏️ fill in what's missing
    print(f"{name} is {age} years old.")

### 🖊️ Classify a handful of Philippine cities

Below is a hand-written list with **approximate** climate numbers (illustrative, not precise data) for a few Philippine cities, in the same order as `classify_koppen`'s parameters — plus Bacolod, using the real numbers you already computed. Loop over the list and print each city's climate code.

In [ ]:
# (city, coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct)
cities = [
    ("Manila",          25.5, 29.5, 27.5, 2000,  15, 85),
    ("Baguio",          15.0, 19.5, 17.5, 3500,  15, 90),
    ("Davao",           26.5, 28.0, 27.2, 1900, 110, 48),
    ("Puerto Princesa", 26.0, 28.5, 27.3, 1450,  45, 72),
    ("Bacolod", coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct),
]

for city, coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct in cities:  # ✏️ loop over cities
    climate_code = classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct)  # ✏️ use your classify_koppen()
    print(f"{city}: {climate_code}")        

---

### Step 4 — loading the gridded present-day climate data

Time to go from a handful of hand-typed cities to *every 0.1° grid point in the Philippines* — 170 latitudes × 135 longitudes. This comes from the same [Beck et al. (2023)](https://doi.org/10.1038/s41597-023-02549-6) dataset as yesterday's Köppen-Geiger maps, but here it's the underlying **monthly climate data** (temperature + precipitation) rather than an already-computed classification — we're about to compute the classification ourselves.

In [ ]:
# Load the dataset
ds_present = xr.open_dataset(PROCESSED_DIR / "climate_philippines_1991_2020_mean.nc")        # ✏️ file: "climate_philippines_1991_2020_mean.nc"

# Inspect it
ds_present

Notice the dimensions: `time` (12 months), `lat` (170), `lon` (135) — this is a genuinely 3D dataset now, unlike yesterday's already-time-averaged maps. `.values` on a variable here gives a `(12, 170, 135)` numpy array: 12 monthly values *for every grid point*.

### 🖊️ Your turn: pull out the two variables as numpy arrays

**Tips:**
- The two variables are `"air_temperature"` and `"precipitation"`
- Use `.values` to get plain numpy arrays (not xarray objects) — easier to index in the loop we're about to write
- Print `.shape` to confirm you get `(12, 170, 135)`

In [ ]:
temp_grid  = ds_present["air_temperature"]        # ✏️ variable "air_temperature". Hint: use `values` to get the raw NumPy array from the xarray DataArray
precip_grid = ds_present["precipitation"]       # ✏️ same for "precipitation"

lats = ds_present["lat"].values              # ✏️ how did we get coordinate values again?
lons = ds_present["lon"].values              # ✏️ how did we get coordinate values again?

💡 Use `shape` to inspect the shape of something:

In [ ]:
print("temp_grid shape:  ", temp_grid.shape)

Try it yourself for `precip_grid`:

In [ ]:
print(precip_grid.shape)                     # ✏️ print the shape of the precipitation grid

> 💬 **Discussion:** Do you understand what the different values in shape mean for our data? Discuss with your neighbours.

---

### Step 5 — classifying the entire grid

To visit every `(lat, lon)` grid point, we need a loop *inside* a loop — a **nested for loop**. 👉 https://wiki.python.org/moin/ForLoop

💡 **Let's learn some things first**:

1. Here's how you make a 2D array (a table):

In [ ]:
my_array = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print(my_array)

# 💡 this array has 3 rows and 3 columns:
print("my_array shape:", my_array.shape)

2. We can index this array in a bunch of ways, look:

In [ ]:
# get the first row
print(my_array[0])

# that's the same as:
print(my_array[0, :])               # 💡 ":" means "all columns"

# get only the first value of the first row
print(my_array[0, 0])

# get the last row
print(my_array[-1])                 # 💡 remember negative indexing if you want stuff at the end!

# get the first column
print(my_array[:, 0])

# get the first value of the first two rows
print(my_array[0:2, 0])             # 💡 use `:` between two indices as 'to'

# ✏️ get the second column of the last two rows
print(my_array[-2:, 1])

3. Now we can understand the example below: a `for` loop *inside* another `for`loop

In [ ]:
# =============================================================
# 💡 EXAMPLE — a nested for loop
# =============================================================
grid = np.array([[1, 2, 3],
                 [4, 5, 6]])

for i in range(grid.shape[0]):           # 💡 English: make i go from 0 to the number of rows
    for j in range(grid.shape[1]):       # 💡 English: make j go from 0 to the number of columns
        print(f"grid[{i},{j}] = {grid[i, j]}")

One more piece of plumbing before the main exercise: `classify_koppen` returns a compressed code (only `Af`/`Am`/`Aw` are full subtypes — B, C, D and E are left as whole groups). To reuse yesterday's 30-class `KG_INFO` colour scheme for our own map, we pick one representative subtype per group to plot with:

In [ ]:
# Standard Köppen-Geiger color scheme (Beck et al. 2023) — same as yesterday
KG_INFO = [
    (1,  "Af",  "#0000FF"), (2,  "Am",  "#0078FF"), (3,  "Aw",  "#46AAFA"),
    (4,  "BWh", "#FF0000"), (5,  "BWk", "#FF9696"), (6,  "BSh", "#F5A500"),
    (7,  "BSk", "#FFD37F"), (8,  "Csa", "#FFFF00"), (9,  "Csb", "#C8C800"),
    (10, "Csc", "#969600"), (11, "Cwa", "#96FF00"), (12, "Cwb", "#64C800"),
    (13, "Cwc", "#329600"), (14, "Cfa", "#C8FF00"), (15, "Cfb", "#64FF00"),
    (16, "Cfc", "#32C800"), (17, "Dsa", "#FF00FF"), (18, "Dsb", "#C800C8"),
    (19, "Dsc", "#963296"), (20, "Dsd", "#966496"), (21, "Dwa", "#AB00AB"),
    (22, "Dwb", "#780078"), (23, "Dwc", "#4B0082"), (24, "Dwd", "#320050"),
    (25, "Dfa", "#00FFFF"), (26, "Dfb", "#37C8FF"), (27, "Dfc", "#007D7D"),
    (28, "Dfd", "#004B4B"), (29, "ET",  "#B2B2B2"), (30, "EF",  "#686868"),
]
cmap = mcolors.ListedColormap([c for _, _, c in KG_INFO])
norm = mcolors.BoundaryNorm(boundaries=np.arange(0.5, 31.5), ncolors=30)

# Our model only outputs group-level codes for B/C/D/E — map each to one representative subtype:
GROUP_TO_KG_CODE = {
    "Af": 1, "Am": 2, "Aw": 3,
    "B": 6,    # representative: BSh (hot semi-arid) — the most plausible arid subtype in the PH
    "C": 11,   # representative: Cwa (humid subtropical, dry winter)
    "D": 26,   # representative: Dfb — included for completeness, unlikely to appear in the PH
    "E": 29,   # representative: ET — included for completeness, unlikely to appear in the PH
}

### 🖊️ Classify every land pixel

We don't want to classify the ocean, instead, we can just assign it `NaN` (**Not a Number**) values. To do so, we create a grid that we first fill completely with NaNs, and then fill with Köppen-Geiger classification where there actually is land.

💡 Using `np.full`, we can make an array filled with some value we want. Here's how you make a 3x3 grid filled with NaNs by combining np.full with `np.nan`:

In [ ]:
my_empty_grid = np.full((3, 3),     # 3 rows, 3 columns
                        np.nan)     # fill with NaN
my_empty_grid

✏️ **Your turn:** make the Köppen-Geiger grid filled with NaNs.

Hint: a couple of cells above you already extracted `lats` and `lons` from `temp_grid`, use `len` to get the right row (lats) and column (lons) sizes for your grid.

In [ ]:
kg_grid_present = np.full((len(lats), len(lons)),           # ✏️ rows, columns = length of latitudes, length of longitudes
                          np.nan)                  # ✏️ NaNs

💡 Cool, we have an ampty grid, now let's fill it with values! First, we have to learn one more thing: when our `for` loop comes across an **ocean pixel**, **we want to skip** it. We can do that with `continue`, look:

In [ ]:
brothers = ["Dries", "Bas", "Daan"]

for brother in brothers:
    if brother == "Bas":        # IF Bas
        continue                # -> we skip to next iteration of the loop
    print(brother)

🖊️ **Let's go!** For every grid point that isn't ocean:
1. Pull out its 12 monthly temperature and precipitation values (`temp_grid[:, i, j]` and `precip_grid[:, i, j]`)
2. Compute the same 6 summary numbers as before
3. Call `classify_koppen(...)` and store the numeric KG code in `kg_grid[i, j]` using `GROUP_TO_KG_CODE`

**Tips:**
- `lsm.values[i, j] == 0` means ocean — `continue` skips straight to the next loop iteration

In [ ]:
# 30
lsm = xr.open_dataset(PROCESSED_DIR / "land_sea_mask_0p1.nc")["land_sea_mask"]  # ✏️ "land_sea_mask_0p1.nc". Hint: one variable -> easiest to use `dataarray`

for i in range(len(lats)):                    # ✏️ how do we make i go from 0 to the number of latitudes?
    for j in range(len(lons)):                # ✏️
        if lsm.values[i, j] == 0:         # ✏️ what value means ocean in the land-sea mask?
            continue                         # ✏️ how do we skip to the next 'j'?

        monthly_T = temp_grid[:, i, j]     # ✏️ temp_grid[:, i, j]
        monthly_P = precip_grid[:, i, j]   # ✏️

        coldest_month_T = float(monthly_T.min())
        warmest_month_T = float(monthly_T.max())
        annual_mean_T   = float(monthly_T.mean())
        annual_precip   = float(monthly_P.sum())
        driest_month_P  = float(monthly_P.min())
        summer_precip_pct = monthly_P[3:9].sum() / monthly_P.sum() * 100             # ✏️ hint: monthly_P[3:9].sum() / monthly_P.sum() * 100

        climate_code = classify_koppen(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct)                  # ✏️ call classify_koppen(...) with the 6 numbers above
        kg_grid_present[i, j] = GROUP_TO_KG_CODE[climate_code]

print("Done! Unique classes found:", sorted(set(kg_grid_present[~np.isnan(kg_grid_present)])))

---

### Step 6 — map it

Same recipe as yesterday: `pcolormesh` with the `KG_INFO` colormap, and a legend built only from the classes actually present.

In [ ]:
shown_present   = [int(v) for v in sorted(np.unique(kg_grid_present[~np.isnan(kg_grid_present)]))]
patches_present = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in shown_present]

fig, ax = plt.subplots(figsize=(7, 9))
ax.pcolormesh(lons, lats, kg_grid_present, cmap=cmap, norm=norm)     # ✏️ x: lons, y: lats, values: kg_grid_present

# ✏️ mark Bacolod (122.95, 10.68), same as yesterday:
ax.scatter(122.95, 10.68, color="black", s=30, marker="*", label="Bacolod")
ax.text(122.95, 10.68, "Bacolod", fontsize=10, ha="left", va="bottom", bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", boxstyle="round,pad=0.2"))

ax.contour(lons, lats, lsm.values, levels=[0.5], colors="black", linewidths=1)

ax.set_title("Our homemade classification — Present (1991-2020)", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
fig.legend(handles=patches_present, loc="lower center", ncol=min(len(patches_present), 6),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Homemade Köppen-Geiger Classification — Philippines", fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])

> 💬 **Discussion:** What do you see? Is Bacolod's own class the one you'd expect from Steps 1-2?

---

### Step 7 — sanity-check against Beck et al.'s own map

How close is our homemade, simplified model to the real thing? Let's plot our map next to `kg_philippines_present.nc` — Beck et al.'s actual 1 km classification from yesterday — side by side.

*(We're only comparing these two maps **by eye** here — a formal pixel-by-pixel agreement score would require regridding one dataset onto the other's resolution, which is a rabbit hole for another day.)*

In [ ]:
# --- Reload Beck et al.'s own present-day classification (same file as yesterday) ---
ds_beck_present = xr.open_dataset(PROCESSED_DIR / "kg_philippines_present.nc")
kg_beck_present = ds_beck_present["kg_class"].squeeze()
kg_beck_present = kg_beck_present.where(kg_beck_present > 0)

shown_beck   = [int(v) for v in sorted(np.unique(kg_beck_present.values)) if not np.isnan(v)]
patches_beck = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in shown_beck]

fig, axes = plt.subplots(1, 2, figsize=(13, 9))

axes[0].pcolormesh(lons, lats, kg_grid_present, cmap=cmap, norm=norm)
axes[0].set_title("Our homemade model (0.1°)", fontsize=12)

axes[1].pcolormesh(kg_beck_present.lon, kg_beck_present.lat, kg_beck_present.values, cmap=cmap, norm=norm)
axes[1].set_title("Beck et al. (2023) (1 km)", fontsize=12)

for ax in axes:
    ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")
axes[0].set_ylabel("Latitude")

all_shown   = sorted(set(shown_present) | set(shown_beck))
all_patches = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in all_shown]
fig.legend(handles=all_patches, loc="lower center", ncol=min(len(all_patches), 8),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Homemade vs. Beck et al. — Present-day Climate Classification", fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.show()

> 💬 **Discussion:** Do the two maps roughly agree? Where do they differ most, and why might that be? *(Hint: think about resolution — 0.1° vs. 1 km — and about everything our simplified model leaves out: no seasonality sub-criteria for C/D, no elevation correction, coarser aridity thresholds.)*

---

### Step 8 — past, present, future

We already have the present-day map. Let's repeat the exact same recipe (Steps 4-6) for the **past** (`climate_philippines_1901_1930_mean.nc`) and **future** (`climate_philippines_2071_2099_ssp460_mean.nc`, under the SSP4-6.0 emissions scenario) — same grid, same land-sea mask, same function.

To avoid retyping the whole loop twice, let's wrap Steps 4-5 into one function of our own: `classify_climate_file`, which takes a filename and returns a finished `kg_grid`.

### 🖊️ Writing `classify_climate_file` from scratch

First step: Let's make a function `get_clim_stats` that can read the .nc files with the climate data and returns the statistics that matter for classification.

Before, we made a nested for loop to go over each pixel one-by-one. However, `xarray` methods are designed specifically for geospatial data, and we can get the same results way easier. Typically, xarray methods allow for you to pass a `dim` (dimension) argument. Look:

In [ ]:
ds = xr.open_dataset(PROCESSED_DIR / "climate_philippines_1991_2020_mean.nc")

# 💡 using the `dim` argument in xarray's `mean` method
mean_temperature = ds["air_temperature"].mean(dim="time")

mean_temperature.plot()

**Boom!** Just like that, you computed the mean in time over the entire data array, no need to write a nested for loop that goes over every pixel. **We can even make this plot in a single line of code:**

In [ ]:
xr.open_dataset(PROCESSED_DIR / "climate_philippines_1991_2020_mean.nc")["air_temperature"].mean(dim="time").plot()

**🖊️ Try it yourself:** plot the climatological precipitation `sum` during **1901-1930** in a **single line**:

In [ ]:
# ✏️ in one line: open climate_philippines_1901_1930_mean.nc, compute the precipitation sum over time, and plot it
xr.open_dataset(PROCESSED_DIR / "climate_philippines_1901_1930_mean.nc")["precipitation"].sum(dim="time").plot()

You're really learning to work with `xarray`, congrats! Now you're ready to **🖊️ create** `get_clim_stats`

**💡 Explanation** about the `filename: str = "climate_philippines_1991_2020_mean.nc"` argument below:
1. `str` specifies that our function expects a string to be input
2. the `= "climate_philippines_1991_2020_mean.nc"` makes this file the default, so that if you don't use any argument at all and just run `get_clim_stats`, it will use that file.

In [ ]:
def get_clim_stats(filename: str = "climate_philippines_1991_2020_mean.nc"):
    
    # 1. load the dataset
    ds = xr.open_dataset(PROCESSED_DIR / filename)

    # 2. extract the temperature and precipitation grids
    temp_grid  = ds["air_temperature"]
    precip_grid = ds["precipitation"]

    # 3. compute the climate statistics
    cm_T = temp_grid.min(dim="time")
    wm_T = temp_grid.max(dim="time")
    am_T = temp_grid.mean(dim="time")
    ap_P = precip_grid.sum(dim="time")
    dm_P = precip_grid.min(dim="time")
    sp_pct = precip_grid[3:9, :, :].sum(dim="time") / precip_grid.sum(dim="time") * 100

    # 4. return the results
    return cm_T, wm_T, am_T, ap_P, dm_P, sp_pct

If you made that correctly, **these two plots should be the same**:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

cm_T, wm_T, am_T, ap_P, dm_P, sp_pct = get_clim_stats()
sp_pct.plot(ax=axes[0], cmap="Blues")

da = xr.open_dataset(PROCESSED_DIR / "climate_philippines_1991_2020_mean.nc")["precipitation"]
sp_pct_2 = da[3:9, :, :].sum(dim="time") / da.sum(dim="time") * 100
sp_pct_2.plot(ax=axes[1], cmap="Blues")

**Beast!** Cool, our `get_clim_stats` reads the file we want, and instantly gives us the 2D climate stats needed for Köppen-Geiger classification. 
Step 3 to get to `classify_climate_file`: we need to adapt our `classify_koppen` function so that it can handle the 2D data. Here, we make a new function `classify_koppen_2D`:

In [ ]:
def classify_koppen_2D(coldest_month_T, warmest_month_T, annual_mean_T, annual_precip, driest_month_P, summer_precip_pct):
    # 💡 no np.asarray this time - we keep everything as xarray DataArrays, so lat/lon coordinates
    # travel along with the result (needed by the two cells after this one). xarray doesn't support
    # 2D boolean *assignment* directly though, so we mutate each DataArray's underlying .values
    # (a plain numpy array) in place instead - the DataArray wrapper (dims, coords) stays intact.

    p_threshold = xr.where(summer_precip_pct >= 70, 20 * annual_mean_T + 280,
                  xr.where(summer_precip_pct <= 30, 20 * annual_mean_T,
                                                     20 * annual_mean_T + 140))

    # Step 2: main group - same reverse-priority mask assignment as before (least important first,
    # so higher-priority conditions get assigned last and "win"), applied to the .values array
    group = xr.full_like(coldest_month_T, "C", dtype="<U1")   # default/"else" case: Temperate
    group.values[(coldest_month_T < -3).values]       = "D"   # Continental
    group.values[(warmest_month_T < 10).values]        = "E"   # Polar
    group.values[(annual_precip < p_threshold).values] = "B"   # Arid
    group.values[(coldest_month_T >= 18).values]       = "A"   # Tropical (checked first -> assigned last -> wins)
    group.values[np.isnan(coldest_month_T.values)]     = ""    # ocean/no-data - not a real class

    # Step 3: subtype (only defined for Group A) - unchanged logic, just applied to every pixel via masks
    is_A = (group == "A").values
    subtype = xr.full_like(coldest_month_T, "", dtype="<U1")
    subtype.values[is_A & (driest_month_P >= 60).values] = "f"                                                    # Af - rainforest
    subtype.values[is_A & (driest_month_P < 60).values & (annual_precip >= 25 * (100 - driest_month_P)).values] = "m"  # Am - monsoon
    subtype.values[is_A & (driest_month_P < 60).values & (annual_precip < 25 * (100 - driest_month_P)).values]  = "w"  # Aw - savanna

    return group + subtype   # 💡 xarray DataArrays support "+" for element-wise string concatenation too, and keep their lat/lon coordinates

**Epic!** Now we can go straight from file to Köppen-Geiger classification with `classify_climate_file`:

In [ ]:
def classify_climate_file(filename: str = "climate_philippines_1991_2020_mean.nc"):
    # takes a filename inside PROCESSED_DIR, returns a (lat, lon) DataArray of numeric KG codes
    cm_T, wm_T, am_T, ap_P, dm_P, sp_pct = get_clim_stats(filename)
    kg_grid = classify_koppen_2D(cm_T, wm_T, am_T, ap_P, dm_P, sp_pct)

    return kg_grid


# --- Use it for past, present (recomputed as a check) and future ---
kg_grid_past    = classify_climate_file("climate_philippines_1901_1930_mean.nc")
kg_grid_present = classify_climate_file("climate_philippines_1991_2020_mean.nc")
kg_grid_future  = classify_climate_file("climate_philippines_2071_2099_ssp460_mean.nc")

print("Past, present and future grids ready.")

In [ ]:
# --- Three-panel map: past, present, future ---
def class_to_code(kg_grid):

    # 💡 map each string climate code to its numeric KG_INFO code, keeping the DataArray/coords intact
    code_grid = xr.full_like(cm_T, np.nan, dtype="float64")
    for code, kg_code in GROUP_TO_KG_CODE.items():
        code_grid.values[(kg_grid == code).values] = kg_code

    return code_grid

grids  = [class_to_code(kg_grid) for kg_grid in [kg_grid_past, kg_grid_present, kg_grid_future]]
titles = ["Past\n1901-1930", "Present\n1991-2020", "Future (SSP4-6.0)\n2071-2099"]

all_shown   = sorted(set(int(v) for g in grids for v in np.unique(g.values[~np.isnan(g.values)])))
all_patches = [mpatches.Patch(color=KG_INFO[i-1][2], label=KG_INFO[i-1][1]) for i in all_shown]

fig, axes = plt.subplots(1, 3, figsize=(16, 8))
for ax, grid, title in zip(axes, grids, titles):
    ax.pcolormesh(lons, lats, grid, cmap=cmap, norm=norm)
    ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")
axes[0].set_ylabel("Latitude")

fig.legend(handles=all_patches, loc="lower center", ncol=min(len(all_patches), 8),
           frameon=False, title="Köppen-Geiger class", fontsize=9)
fig.suptitle("Homemade Köppen-Geiger Classification — Philippines, Past/Present/Future", fontsize=13)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

### Step 9 — wrap-up: zooming back into Bacolod

Let's check numerically whether our own model predicts Bacolod's climate class changing over time. We need the grid indices closest to Bacolod's coordinates (122.95°E, 10.68°N):

In [ ]:
# 💡 np.argmin(np.abs(...)) finds the index of the closest value in an array
bacolod_i = np.argmin(np.abs(lats - 10.68))
bacolod_j = np.argmin(np.abs(lons - 122.95))

for grid, label in zip(grids, titles):
    code = int(grid[bacolod_i, bacolod_j])
    print(f"{label.splitlines()[0]:9s}: {KG_INFO[code - 1][1]}")

> 💬 **Wrap-up discussion:**
> - What changes across the three maps, broadly? Does the shift match what Beck et al.'s own past→future comparison showed yesterday?
> - Does your homemade model predict a change in Bacolod's own climate class? If so, from what to what — and what would that mean for agriculture, water supply, or biodiversity in Negros?
> - You've now built, from scratch, a full pipeline: raw gridded climate data → a `def` function encoding real scientific criteria → a nested `for` loop applying it everywhere → a map. That's the core pattern behind a huge amount of real climate science.

---

## Well done — Part I complete!

You just:
- Learned `def` functions and `return`
- Learned `for` loops, including nested loops over a 2D grid
- Rewrote yesterday's hand-classification as a reusable, scientifically rigorous function
- Classified the entire Philippines, at three points in time, from raw gridded climate data
- Compared your homemade model against the real Beck et al. (2023) product

Next: **Part II**, where we shift from *climate* to the *life* that climate shapes — vegetation, satellite imagery, and the animals of Negros.

## *PART II: The Philippines as a Biodiversity Hotspot*

### Vegetation types & forest cover

Yesterday and this morning, we classified climate zones using [Köppen's system](https://en.wikipedia.org/wiki/K%C3%B6ppen_climate_classification) — and Köppen, originally a botanist, designed his thresholds specifically so that climate zones would line up with **vegetation zones** (Af regions look like rainforest, Aw regions look like savanna, and so on). Today we look at *actual* vegetation data instead of using climate as a proxy for it.

We'll **zoom into Negros** and use the **[Hansen Global Forest Change dataset](https://storage.googleapis.com/earthenginepartners-hansen/GFC-2023-v1.11/download.html)** (Hansen et al., 2013, *Science* — 👉 https://doi.org/10.1126/science.1244693), which maps global tree cover and forest loss from satellite imagery at 30 m resolution. You can browse it interactively yourself at 👉 https://glad.earthengine.app/view/global-forest-change.

For now we only need the `treecover2000` variable (percent canopy cover in the year 2000) from `negros_forest_change.nc` — we'll use `lossyear` (year of first forest loss) later, in Part IV.

In [ ]:
# --- Load Negros forest cover data ---
forest_ds = xr.open_dataset(PROCESSED_DIR / "day2" / "negros_forest_change.nc")
treecover = forest_ds["treecover2000"]

treecover

💡 this file is native 30m resolution (~4800 x 4400 pixels) - **we'll coarsen it** a bit so maps stay fast to draw:

In [ ]:
treecover_coarse = treecover.coarsen(latitude=10, longitude=10, boundary="trim").mean()
treecover_coarse

**🖊️ Plot the tree cover map and mark Bacolod.**

Bacolod marker at (122.95°E, 10.68°N), same as you've done on previous maps.

In [ ]:
# ✏️ visualize the tree cover data for Negros and mark Bacolod:
fig, ax = plt.subplots(figsize=(7, 7))
treecover_coarse.plot(ax=ax, cmap="Greens", vmin=0, vmax=100,
                       cbar_kwargs={"label": "Tree cover 2000 (%)"})
ax.scatter(122.95, 10.68, color="black", s=60, zorder=5)
ax.text(122.97, 10.69, "Bacolod", fontsize=9)
ax.set_title("Tree Cover — Negros (year 2000)")
ax.set_aspect("equal")

**🖊️ Summary statistics:**

Using the full-resolution `treecover` (not the coarsened version, so the percentage is precise), compute:
1. The mean tree cover % over all of Negros
2. The percentage of pixels with more than 50% tree cover (a common cutoff for "forest")

In [ ]:
mean_cover = float(treecover.mean())            # ✏️ mean tree cover % over Negros
frac_forest = float((treecover > 50).mean())           # ✏️ fraction of pixels with treecover > 50%

print(f"Mean tree cover over Negros: {mean_cover:.1f}%")
print(f"Pixels with >50% tree cover: {frac_forest * 100:.1f}%")

> 💬 **Discussion:** The mean is lower than you might expect for a "forested tropical island" — why? *(Hint: what area are we taking the mean over?)* Where on the map is Negros most forested? Does that line up with the wetter (Af/Am) vs. drier (Aw) zones from Part I's homemade climate map?

---

### Satellite imagery

What does a satellite actually "see"? Not a photograph — a grid of numbers, one per pixel, for each of several **spectral bands** (wavelength ranges). Some of those bands aren't even visible to our eyes.

We'll use the **[Sentinel-2](https://sentinel.esa.int/web/sentinel/missions/sentinel-2)** mission (ESA/Copernicus): two satellites launched in 2015 and 2017 that jointly revisit every point on Earth every ~5 days, at 10-60 m resolution. `negros_sentinel2.nc` has two cloud-free mosaics over Negros — 2018 ("old") and 2024 ("new") — with these bands:
- `red`, `nir` (near-infrared) — raw reflectance values. That is: the satellite beams red and near-infrared to the Earth and measures how much reflects back.
- `visual_r`, `visual_g`, `visual_b` — a ready-made true-colour image (what your eyes would actually see)

*(Why 2018 and not, say, 2000? Sentinel-2 only launched in 2015 — before that, this specific satellite simply didn't exist. 2018 is close to the earliest point with reliably cloud-free coverage over Negros.)*

A single band is just a 2D array of numbers — let's look at one directly:

In [ ]:
# --- Load Sentinel-2 data ---
s2_ds = xr.open_dataset(PROCESSED_DIR / "day2" / "negros_sentinel2.nc")
print(s2_ds.time.values)   # the two available dates

# =============================================================
# 💡 EXAMPLE — a single band is just numbers
# =============================================================
red_new = s2_ds["red"].sel(time="2024-04-03")

fig, ax = plt.subplots(figsize=(6, 6))
red_new.plot(ax=ax, cmap="gray")
ax.set_title("Red band only — just one number per pixel")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> **Discussion:** What do you notice about the red band? What type of surface reflects a lot of red light?

**🖊️ Data quality check:** what percentage of values in `red_new` are NaNs?

Hints: remember xarray's `shape`, `np.isnan` and `np.sum`

In [ ]:
# ✏️ Compute the percentage of pixels that has a missing value
# Hint: number of NaNs / total number of pixels * 100
np.sum(np.isnan(red_new)) / np.prod(red_new.shape) * 100

To get an actual photo-like image, we need to combine three bands (red, green, blue) into a single array of shape `(rows, cols, 3)` — one extra dimension holding the 3 colour channels for every pixel. `np.dstack` stacks 2D arrays along a new third axis to do exactly this:

In [ ]:
# =============================================================
# 💡 EXAMPLE — stacking 3 bands into a true-colour image
# =============================================================
s2_new = s2_ds.sel(time="2024-04-03")

# 💡 a few pixels are NaN - fill with 0 (black) so imshow doesn't complain
r = np.nan_to_num(s2_new["visual_r"].values)
g = np.nan_to_num(s2_new["visual_g"].values)
b = np.nan_to_num(s2_new["visual_b"].values)

rgb_new = np.dstack([r, g, b]).astype(np.uint8)   # shape (rows, cols, 3)
print("rgb_new shape:", rgb_new.shape)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb_new)          # 💡 imshow displays a (rows, cols, 3) array directly as a photo
ax.set_title("True-colour image — Negros, 2024")
ax.axis("off")
plt.tight_layout()
plt.show()

### 🖊️ Your turn: old vs. new, side by side

Build the same kind of true-colour image for the **2018** ("old") time slice, and plot it next to the 2024 image you already have — a two-panel figure, same subplot skills as yesterday.

**Tips:**
- The "old" date is `"2018-02-09"`
- Reuse `rgb_new` (already built above) for the right-hand panel

In [ ]:
# --- Build the "old" (2018) true-colour image ---
s2_old = s2_ds.sel(time="2018-02-09")          # ✏️ select the 2018 date

r_old = np.nan_to_num(s2_old["visual_r"].values)
g_old = np.nan_to_num(s2_old["visual_g"].values)                           # ✏️ same as above, for green
b_old = np.nan_to_num(s2_old["visual_b"].values)                           # ✏️ same as above, for blue

rgb_old = np.dstack([r_old, g_old, b_old]).astype(np.uint8)                         # ✏️ np.dstack([...]).astype(np.uint8)

# --- Two-panel comparison ---
fig, axes = plt.subplots(1, 2, figsize=(13, 7))

axes[0].imshow(rgb_old)                   # ✏️ rgb_old
axes[0].set_title("2018")

axes[1].imshow(rgb_new)                   # ✏️ rgb_new
axes[1].set_title("2024")

for ax in axes:
    ax.axis("off")

fig.suptitle("Negros, Sentinel-2 true colour — 2018 vs. 2024")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** What visible changes can you spot between 2018 and 2024, just by eye? Can you point out forest, farmland, urban areas, and coastline in the images — and does what you see line up with what you'd expect from the `treecover2000` map, and from Bacolod's known location?

---

### Fauna

Climate and vegetation set the stage — now let's look at who actually lives there. We'll use occurrence records from **[GBIF](https://www.gbif.org/)** (the Global Biodiversity Information Facility), a free, public database aggregating millions of species sightings and museum specimens worldwide from thousands of institutions.

Two flagship species first:
- **[Philippine Eagle](https://en.wikipedia.org/wiki/Philippine_eagle)** (*Pithecophaga jefferyi*) — one of the world's largest and rarest eagles, endemic to the Philippines, Critically Endangered, and entirely forest-dependent. 👉 https://www.gbif.org/species/2480381
- **[Large flying fox](https://en.wikipedia.org/wiki/Large_flying_fox)** (*Pteropus vampyrus*) — one of the largest bats in the world, found across the Philippine archipelago. 👉 https://www.gbif.org/species/5218644

In [ ]:
# --- Load occurrence records ---
eagle = pd.read_csv(PROCESSED_DIR / "day2" / "gbif_philippine_eagle.csv")
flying_fox = pd.read_csv(PROCESSED_DIR / "day2" / "gbif_flying_fox.csv")

print(f"Philippine Eagle records: {len(eagle)}")
print(f"Flying fox records: {len(flying_fox)}")
eagle.head()

### 🖊️ Your turn: map the occurrence points

No basemap needed — a plain `ax.scatter(lon, lat, ...)` is enough, and we can reuse yesterday's `land_sea_mask.nc` contour lines as a rough Philippines outline, exactly like you did for the temperature/precipitation maps.

Make a two-panel figure: eagle records on the left, flying fox records on the right.

In [ ]:
lsm_ph = xr.open_dataarray(PROCESSED_DIR / "land-sea-mask_0p00833333.nc")

fig, axes = plt.subplots(1, 2, figsize=(12, 8))

axes[0].scatter(eagle["lon"], eagle["lat"], color="firebrick", s=20, alpha=0.7)     # ✏️ eagle["lon"], eagle["lat"]
axes[0].set_title(f"Philippine Eagle ({len(eagle)} records)")

axes[1].scatter(flying_fox["lon"], flying_fox["lat"], color="steelblue", s=20, alpha=0.7)     # ✏️ flying_fox["lon"], flying_fox["lat"]
axes[1].set_title(f"Flying Fox ({len(flying_fox)} records)")

for ax in axes:
    ax.contour(lsm_ph.lon, lsm_ph.lat, lsm_ph.values,
               levels=[0.5], colors=["#666666"], linewidths=0.8, alpha=0.7)
    ax.set_xlabel("Longitude")
    ax.set_aspect("equal")
axes[0].set_ylabel("Latitude")

plt.tight_layout()
plt.show()

> 💬 **Discussion:** What patterns do you notice in the observation records of the Philippine Eagle and flying fox? Given that the eagle is entirely forest-dependent and Critically Endangered, what does that clustering tell you? *(Keep this in mind — we'll come back to forest loss and its effect on species like this in Part IV.)*

---

Two species is a start, but let's look at a much richer picture: **every terrestrial mammal record for the Philippines** in GBIF.

In [ ]:
# --- Load the full mammal community dataset ---
mammals = pd.read_csv(PROCESSED_DIR / "day2" / "gbif_mammals_philippines.csv")
print(f"{len(mammals)} records, {mammals['species'].nunique()} species")
mammals.head()

### 🖊️ Your turn: which mammal groups dominate?

Use `.value_counts()` on the `order` column to see which taxonomic groups have the most records. There's a surprise waiting.

In [ ]:
order_counts = mammals["order"].value_counts()     # ✏️ mammals["order"].value_counts()
print(order_counts)

> 💬 **Discussion:** Which order dominates, and by how much? Does that match what you'd naively guess about "Philippine mammals"? *(Hint: think about how easy each group is to survey. Record counts reflect sampling effort as much as true abundance!)*

---

### Data quality check

Before we filter and plot, let's check how complete the columns we're about to rely on actually are — real-world biodiversity data is rarely 100% populated, and it's good practice to check *before* you filter on a column, not after.

In [ ]:
# =============================================================
# 💡 EXAMPLE — checking how many values are missing in a column, here "county"
# =============================================================
print(mammals["county"].isnull().sum(), "missing out of", len(mammals))          # count of missing values
print(f"{mammals['county'].isnull().mean()*100:.1f}% missing")                                        # 💡 same trick as before: True=1/False=0, so .mean() = fraction missing

**🖊️ check null rates** on the columns we're about to use

We're about to filter records to Negros using `stateProvince`, and later make a bar chart of `iucnRedListCategory`. Check the missing-value fraction for both (plus `family` and `order`, for comparison, since we know those are complete).

In [ ]:
columns_to_check = ["stateProvince", "iucnRedListCategory", "family", "order"]

for col in columns_to_check:
    frac_missing = mammals[col].isnull().mean()     # ✏️ mammals[col].isnull().mean()
    print(f"{col:20s}: {frac_missing * 100:5.1f}% missing")

> 💬 **Discussion:** `stateProvince` and `iucnRedListCategory` both have a meaningful chunk of missing values, while `family`/`order` are essentially complete. What does that mean for the filtering and bar chart we're about to make? *(We'll simply be working with a subset of all records — not a bias-free sample, but still informative.)*

---

### Filtering to Negros

`stateProvince` isn't perfectly clean text — a look at the raw values shows entries like `"Negros Oriental"`, `"Negros Occidental"`, `"Negros I"`, `"Negros Id."`. Rather than matching one exact string, we can check whether `"Negros"` appears *anywhere* in the text using `.str.contains()`:

In [ ]:
# =============================================================
# 💡 EXAMPLE — filtering rows by a partial text match
# =============================================================
sample_values = pd.Series(["Negros Oriental", "Negros Occidental", "Cebu", "Negros I", np.nan])

is_negros = sample_values.str.contains("Negros", case=False, na=False)   # 💡 na=False: treat missing values as "no match", not an error
print(is_negros)

### 🖊️ Your turn: filter the mammal records to Negros, and map them

**Tips:**
- Same pattern as the example, applied to `mammals["stateProvince"]`
- Use `mammals[is_negros]` to keep only the matching rows (same boolean-filtering idea as yesterday's `df[condition]`)
- Reuse the scatter + `land_sea_mask.nc` contour pattern from the eagle/flying fox map — but this time, zoom the axes to Negros using `ax.set_xlim(122.3, 123.4)` and `ax.set_ylim(9.8, 11.0)` (the same bounding box as the Sentinel-2/Hansen data)

In [ ]:
is_negros = mammals["stateProvince"].str.contains("Negros", case=False, na=False)                    # ✏️ mammals["stateProvince"].str.contains("Negros", case=False, na=False)
negros_mammals = mammals[is_negros]               # ✏️ mammals[is_negros]
print(f"{len(negros_mammals)} mammal records on Negros")

fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(negros_mammals["decimalLongitude"], negros_mammals["decimalLatitude"],
           color="darkorange", s=10, alpha=0.5)
ax.contour(lsm_ph.lon, lsm_ph.lat, lsm_ph.values,
           levels=[0.5], colors=["#666666"], linewidths=0.8, alpha=0.7)

ax.set_xlim(122.3, 123.4)          # ✏️ zoom to Negros
ax.set_ylim(9.8, 11.0)
ax.set_title(f"Mammal occurrence records — Negros (total: {len(negros_mammals)})")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")

> 💬 **Discussion:** Notice the points scattered far outside Negros itself (even outside the Philippines!) if you check `negros_mammals["decimalLongitude"].describe()` — a good reminder that a `stateProvince` text match isn't a perfect filter; some records are simply mislabeled or mis-geocoded. Real biodiversity data always needs this kind of scrutiny.

### 🖊️ Conservation status bar chart

Let's check how many animals belong to the different categories! We'll make a bar chart of `iucnRedListCategory` value counts for the Negros mammal subset.

**Tip:**
- `.value_counts()` on `negros_mammals["iucnRedListCategory"]` gives you counts per category directly

In [ ]:
iucn_counts = negros_mammals["iucnRedListCategory"].value_counts()    # ✏️ negros_mammals["iucnRedListCategory"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(iucn_counts.index, iucn_counts.values, color="darkorange")   # ✏️ iucn_counts.index, iucn_counts.values

ax.set_title("IUCN Red List Category — Negros mammal records")
ax.set_xlabel("Category")
ax.set_ylabel("Number of records")

**🖊️ Look up** some of the specific `species` behind the EN/CR/VU records on Negros (`negros_mammals[negros_mammals["iucnRedListCategory"].isin(["EN", "CR", "VU"])]["species"].unique()`). What species are they? What threatens them?

In [ ]:
negros_mammals[negros_mammals["iucnRedListCategory"].isin(["EN", "CR", "VU"])]["species"].unique()      # ✏️

> 💬 **Final discussion:** The **[IUCN Red List](https://www.iucnredlist.org/)** categorises extinction risk from least to most severe: **LC** (Least Concern) → **NT** (Near Threatened) → **VU** (Vulnerable) → **EN** (Endangered) → **CR** (Critically Endangered), plus **DD** (Data Deficient — not enough information to assess).
> - Most records are LC, which makes sense — common species get recorded more often. But which categories appear at all for Negros, and what does that tell you about the mammals sharing the island?
> - What vulnerable, endangered and critically endangered species did we find in this dataset? What do you think are the causes for their conservation status?
> - Keep this in mind for Parts III-IV: we're about to map land use and its change over time from satellite imagery!

---

## Well done — Part II complete!

You just:
- Connected yesterday's climate classification to actual vegetation and forest cover data
- Learned to build true-colour images from raw satellite bands (`np.dstack`)
- Compared satellite imagery across time, by eye
- Worked with real, messy biodiversity occurrence data (GBIF): filtering, null-checking, and conservation-status reporting

Next: **Part III**, where we go from satellite imagery to an actual land use classification.

# CAS: make an additional exercise. Something like unique species count in a pixel ~ biodiversity, or alternatively, plot the distribution of endangered species, something like that.
### Idea: regridding species observations to 1km grid -> summing unique species observed in grid cells -> "biodiversity" -> possibly train linear model on environmental data and predict biodiversity on unseen pixels -> verify against student's knowledge?

## *PART III: From satellite imagery to land use*

Köppen designed his climate thresholds around vegetation zones; now let's flip that around and classify *land cover itself* directly from the Sentinel-2 imagery you already loaded in Part II — using the exact same recipe as Part I's homemade climate classifier: turn a decision rule into a `def` function, apply it to a whole grid at once, and map the result.

### 1) NDVI: turning two bands into one meaningful number

**Healthy, lush-green vegetation absorbs a lot of red light but reflects a lot of near-infrared**. Remember that we had these two bands available in our Sentinel-2 imagery? We can combine them in the [Normalized Difference Vegetation Index NDVI](https://en.wikipedia.org/wiki/Normalized_difference_vegetation_index), the most widely used metric to derive vegetation status from remote sensing data. NDVI is computed as:

**NDVI = (NIR − Red) / (NIR + Red)**

NDVI ranges from -1 to +1: strongly **negative** for water, **near zero** for bare soil/urban surfaces, and **high** (often >0.3) for healthy vegetation. Let's write a small reusable function for it:

In [ ]:
def compute_ndvi(red, nir):
    return (nir - red) / (nir + red)


# --- Apply it to the 2024 scene (s2_new, from Part II) and map it ---
ndvi_new = compute_ndvi(s2_new["red"], s2_new["nir"])

fig, ax = plt.subplots(figsize=(7, 7))
ndvi_new.plot(ax=ax, cmap="RdYlGn", vmin=-1, vmax=1, cbar_kwargs={"label": "NDVI"})
ax.set_title("NDVI — Negros, 2024")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Which parts of the map are reddish (low/negative NDVI)? Which are green (high NDVI)? Do the reddish areas match what you'd expect for water and urban areas from the true-colour image earlier?

---

### 2) From NDVI to land-cover classes

Let's turn NDVI into actual land-cover classes using a simple, interpretable rule — deliberately similar in spirit to Köppen's group logic: a handful of threshold conditions on physically meaningful numbers.

Looking at real values in this scene: NDVI ranges roughly -0.95 to +0.97 (mean ~0.32 in 2018, ~0.26 in 2024), and *brightness* (the average of red and nir) is much lower over water than over land or bare ground. That gives us three classes:

| Class | Rule |
|---|---|
| **Water** | low brightness *and* low NDVI (dark in both bands — water absorbs rather than reflects) |
| **Vegetation** | high NDVI |
| **Bare/Urban** | everything else (moderate-to-high brightness, low NDVI) |

We'll use `brightness < 800` and `NDVI < 0.1` for Water, and `NDVI > 0.3` for Vegetation — thresholds picked by eye against this scene's own numbers.

### 🖊️ Your turn: `classify_landcover_2D`

Reuse the exact "boolean-mask, reverse-priority overwrite" trick from `classify_koppen_2D` in Part I — same technique, brand new domain, xarray structure preserved throughout.

In [ ]:
def classify_landcover_2D(red, nir):
    
    ndvi = compute_ndvi(red, nir)              # ✏️ reuse the function you just wrote
    brightness = (red + nir) / 2

    landcover = xr.full_like(red, "Bare/Urban", dtype="<U10")               # ✏️ default/"else" case
    landcover.values[(brightness < 800).values & (ndvi < 0.1).values] = "Water"        # ✏️ dark in both bands
    landcover.values[(ndvi > 0.3).values] = "Vegetation"                               # ✏️ high NDVI, checked last -> wins
    landcover.values[np.isnan(red.values)] = ""                                        # ✏️ cloud/no-data gaps - not a real class

    return landcover


# --- Try it on the 2024 scene ---
landcover_new = classify_landcover_2D(s2_new["red"], s2_new["nir"])
print(dict(zip(*np.unique(landcover_new.values, return_counts=True))))

Now let's map it, using its own small categorical colour scheme — same recipe as the Köppen-Geiger maps:

In [ ]:
LC_INFO = [
    (1, "Water",      "#3B4CC0"),
    (2, "Vegetation", "#2E8B57"),
    (3, "Bare/Urban", "#C2A878"),
]
lc_cmap = mcolors.ListedColormap([c for _, _, c in LC_INFO])
lc_norm = mcolors.BoundaryNorm(boundaries=np.arange(0.5, len(LC_INFO) + 1.5), ncolors=len(LC_INFO))
LC_TO_CODE = {name: code for code, name, _ in LC_INFO}


def landcover_to_code(landcover):
    # 💡 same trick as class_to_code in Part I: map each string label to its numeric code, keep the DataArray/coords intact
    code_grid = xr.full_like(landcover, np.nan, dtype="float64")
    for name, code in LC_TO_CODE.items():
        code_grid.values[(landcover == name).values] = code
    return code_grid


fig, ax = plt.subplots(figsize=(7, 7))
code_new = landcover_to_code(landcover_new)
ax.pcolormesh(code_new.longitude, code_new.latitude, code_new, cmap=lc_cmap, norm=lc_norm)
ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
ax.set_title("Homemade land cover — Negros, 2024")
ax.set_aspect("equal")
patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
ax.legend(handles=patches, loc="lower left", frameon=True, fontsize=9)
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Compare this to the true-colour image from Part II — does Water/Vegetation/Bare-Urban line up with what you saw by eye?

---

### 3) Wrap into one reusable function

Same shape as Part I's `classify_climate_file`: load a file, compute what's needed, classify, return.

In [ ]:
def classify_negros_landcover(time="2024-04-03"):
    # takes a Sentinel-2 date, returns a (latitude, longitude) DataArray of land-cover labels
    s2 = xr.open_dataset(PROCESSED_DIR / "day2" / "negros_sentinel2.nc").sel(time=time)
    return classify_landcover_2D(s2["red"], s2["nir"])


# --- Use it for both available dates ---
landcover_2018 = classify_negros_landcover("2018-02-09")
landcover_2024 = classify_negros_landcover("2024-04-03")

print("2018:", dict(zip(*np.unique(landcover_2018.values, return_counts=True))))
print("2024:", dict(zip(*np.unique(landcover_2024.values, return_counts=True))))

Map both side by side:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

for ax, landcover, title in zip(axes, [landcover_2018, landcover_2024], ["2018", "2024"]):
    code_grid = landcover_to_code(landcover)                                                        # ✏️ landcover_to_code(landcover)
    ax.pcolormesh(code_grid.longitude, code_grid.latitude, code_grid, cmap=lc_cmap, norm=lc_norm)   # ✏️ pcolormesh with lc_cmap/lc_norm
    ax.scatter(122.95, 10.68, color="black", s=30, zorder=5)                                        # ✏️ mark Bacolod
    ax.set_title(title)
    ax.set_aspect("equal")

patches = [mpatches.Patch(color=color, label=name) for _, name, color in LC_INFO]
fig.legend(handles=patches, loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Homemade Land Cover — Negros, 2018 vs. 2024")
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

> **Discussion:** What do you notice? Do you think our data is appropriate to accurately visualize land use classes? Is the Bare-Urban class reliable? **What's going wrong?**

Let's sanity-check our 2024 classification against the real `treecover2000` map from Part II — side by side, purely by eye (different sensor, different resolution, so no pixel-by-pixel score, same design choice as Part I Step 7):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 8))

code_2024 = landcover_to_code(landcover_2024)
axes[0].pcolormesh(code_2024.longitude, code_2024.latitude, code_2024, cmap=lc_cmap, norm=lc_norm)
axes[0].set_title("Our homemade land cover (2024)")

treecover.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[1].set_title("Hansen tree cover (2000)")

for ax in axes:
    ax.set_aspect("equal")

> **Discussion:** As you can see, it's not all that straightforward to make a tree cover map as beautiful as that of Hansen. Although we're doing something else - land use classification - *working with satellite data is clearly a tricky business.*

---

## Well done — Part III complete!

You just:
- Learned what NDVI is and why it works (red absorption vs. NIR reflection in healthy vegetation)
- Reused Part I's exact "rule -> vectorized function -> map" pattern in a brand new domain
- Classified real satellite imagery into land-cover classes, for two points in time
- Wrapped the whole pipeline into one reusable function, ready to reuse in Part IV

Next: **Part IV**, where we put 2018 and 2024 side by side to actually measure what changed.

## *PART IV: Land Use Change*

Two independent looks at the same six years of change on Negros: Hansen's own annual forest-loss labels, and change detected from *our own* homemade land-cover classifier from Part III — cross-checked against each other, the same way Part I cross-checked its homemade climate map against Beck et al.

### 1) Forest cover loss

`negros_forest_change.nc` has a second variable we haven't used yet: `lossyear` — the year each pixel first lost forest cover (1 = 2001, ..., 23 = 2023, 0 = no loss detected). Let's load it:

In [ ]:
lossyear = forest_ds["lossyear"]        # already opened in Part II as forest_ds
lossyear

### 🖊️ How many pixels were lost each year?

`np.unique(..., return_counts=True)` is the numpy equivalent of pandas' `.value_counts()` from Part II, applied to a plain array:

In [ ]:
years, counts = np.unique(lossyear.values, return_counts=True)     # ✏️ np.unique(..., return_counts=True)

for year_code, count in zip(years, counts):
    if year_code == 0:
        continue                        # 0 = no loss detected, not an actual year
    print(f"{2000 + int(year_code)}: {count} pixels")

**Bar plot** of pixels lost per year:

In [ ]:
loss_only = counts[years != 0]
years_only = 2000 + years[years != 0].astype(int)

fig, ax = plt.subplots(figsize=(11, 4))

ax.bar(years_only, loss_only, color="firebrick")           # ✏️ years_only, loss_only

ax.set_title("Forest Loss per Year — Negros (Hansen et al.)")
ax.set_xlabel("Year")
ax.set_ylabel("Pixels lost (30m)")

> 💬 **Discussion:** Any standout years? *(Real data shows a clear spike around 2018, and again in 2022-2023 — any guesses why, before we tell you? Think El Niño-driven dry seasons/fires, logging booms, or just natural year-to-year noise.)*

Map `lossyear` spatially (masking the "no loss" pixels so they don't dominate the colour scale), next to `treecover2000`:

In [ ]:
forest_ds

In [ ]:
lost_any = (lossyear > 0).astype("int8")   # 1 = forest lost at least once, 0 = no loss

fig, axes = plt.subplots(1, 2, figsize=(13, 8))

lsm_negros = xr.open_dataarray(PROCESSED_DIR / "land-sea-mask_0p00833333.nc").sel(
    lon=slice(float(treecover.longitude.min()), float(treecover.longitude.max())),
    lat=slice(float(treecover.latitude.max()), float(treecover.latitude.min())),
)

cmap_bin = mcolors.ListedColormap(["white", "red"])  # 0 -> white, 1 -> red
norm_bin = mcolors.BoundaryNorm([-0.5, 0.5, 1.5], cmap_bin.N)

lost_any.plot(
    ax=axes[0],
    cmap=cmap_bin,
    norm=norm_bin,
    cbar_kwargs={"ticks": [0, 1], "label": "Forest loss (0/1)"},
)
axes[0].set_title("Forest loss: binary")

treecover.plot(ax=axes[1], cmap="Greens", vmin=0, vmax=100, add_colorbar=False)
axes[1].set_title("Tree cover (2000)")

for ax in axes:
    ax.contour(
        lsm_negros.lon, lsm_negros.lat, lsm_negros.values,
        levels=[0.5], colors=["#666666"], linewidths=0.8, alpha=0.7
    )
    ax.set_xlim(float(treecover.longitude.min()), float(treecover.longitude.max()))
    ax.set_ylim(float(treecover.latitude.min()), float(treecover.latitude.max()))
    ax.set_aspect("equal")

> 💬 **Discussion:** Where has loss concentrated? Does it look random, or clustered along certain patterns (roads, forest edges, specific regions)?

### 🖊️ Quantify loss over the 2018-2023 window

This is the same window our two Sentinel-2 snapshots span — setting up the cross-check in subsection 2:

In [ ]:
had_forest_2000 = treecover > 25                        # ✏️ treecover > 25 (a simple "forest" baseline threshold)
lost_2018_2023 = (lossyear >= 17) & (lossyear <= 23)    # ✏️ lossyear codes 17-23 = calendar years 2018-2023

frac_lost = float((lost_2018_2023 & had_forest_2000).sum() / had_forest_2000.sum())
print(f"Fraction of Negros' 2000 forest baseline lost between 2018-2023: {frac_lost * 100:.1f}%")

---

### 2) Change detection from our own classifier

We already have `landcover_2018` and `landcover_2024` from Part III. Let's write one reusable function to detect any transition between classes — general enough to answer both "where did we lose vegetation?" (this subsection) and "where did we gain urban?" (next subsection) with the very same function:

In [ ]:
def detect_change(before, after, from_class=None, to_class=None):
    # ✏️ from_class/to_class are optional: None means "any class". A pixel counts as changed if
    # it was `from_class` before, is `to_class` after, and the two dates actually differ.
    has_data  = (before != "") & (after != "")            # ✏️ ignore cloud/no-data gaps in either year!
    was_from  = True if from_class is None else (before == from_class)
    became_to = True if to_class   is None else (after == to_class)
    return has_data & was_from & became_to & (before != after)


lost_vegetation = detect_change(landcover_2018, landcover_2024, from_class="Vegetation")
print(f"Pixels that were Vegetation in 2018 but aren't anymore: {int(lost_vegetation.sum())}")

💡 Note: without the `has_data` check, the huge cloud-gap difference between the two dates (~21% of the 2018 scene is no-data, vs. well under 1% in 2024) would get counted as spurious "change" — always account for missing data before comparing two time periods!

Map the lost-vegetation mask:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.pcolormesh(lost_vegetation.longitude, lost_vegetation.latitude, lost_vegetation,
              cmap=mcolors.ListedColormap(["#eeeeee", "firebrick"]))
ax.scatter(122.95, 10.68, color="black", s=30, zorder=5)
ax.set_title("Lost vegetation, 2018 -> 2024 (our own classifier)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

frac_lost_ours = float(lost_vegetation.sum()) / int((landcover_2018 == "Vegetation").sum())
print(f"Fraction of 2018's Vegetation-classified pixels that changed by 2024: {frac_lost_ours * 100:.1f}%")

> 💬 **Discussion:** Our own classifier says roughly **18%** of 2018's vegetated pixels changed class by 2024 — much higher than Hansen's own **~1%** loss estimate for the same window (subsection 1)! Why such a big gap? A few real reasons:
> - The two Sentinel-2 dates aren't the same time of year (Feb vs. April) — NDVI naturally shifts with the wet/dry season cycle, independent of any real land-use change.
> - Our "Vegetation" class lumps together forest *and* farmland/grass, while Hansen's `lossyear` specifically tracks forest.
> - Our classifier works on single snapshots with one crude threshold; Hansen's product is built from many years of imagery per pixel.
>
> This is a genuine lesson from real remote sensing: a quick homemade classifier can dramatically overestimate change if you're not careful about *when* your snapshots were taken.

---

### 3) Let's fix it: redo the comparison with proper composite data

The 18% number above isn't really measuring land-use change — it's mostly clouds and a Feb-vs-April seasonal mismatch. Rather than just diagnosing that and moving on, `day2_bts.ipynb` already built the fix: instead of one snapshot per date, pool **many** scenes over the same **fixed dry-season window** across **several years**, mask clouds/shadows per-pixel using each sensor's own QA band, and take the per-pixel **temporal median**. The result is saved as `negros_imagery.nc` — a historical composite (Landsat 5 TM, 133 scenes pooled from Feb-Apr 1988-2000) and a recent composite (Sentinel-2 L2A, 156 scenes pooled from Feb-Apr 2022-2024).

**One honest difference before we start:** this isn't the same 2018-2024 window — it's roughly 1996 vs. 2023, a much longer span. Sentinel-2 only launched in 2015, so reaching further back means switching sensors entirely (to Landsat 5). We're not redoing the exact same measurement with better data; we're making a different, longer, but properly-engineered measurement instead.

In [ ]:
img = xr.open_dataset(PROCESSED_DIR / "day2" / "negros_imagery.nc")
print(img.time.values)

for label, t in [("1996 (historical)", "1996-03-15"), ("2023 (recent)", "2023-03-15")]:
    nan_frac = float(np.isnan(img["red"].sel(time=t).values).mean())
    print(f"{label}: {nan_frac:.1%} no-data")

> 💬 **Discussion:** The recent composite is essentially gap-free now (vs. well under 1% before — already good, barely moves). The historical composite is a different story: **~31% no-data**, *worse* than the original single 2018 scene's ~21%! Pooling *more* scenes doesn't automatically mean *fewer* gaps — it means more honest bookkeeping about where *no* scene in 1988-2000 ever caught a clear view. Older imagery is scarcer and noisier, full stop; properly accounting for missing data doesn't make it disappear, it just stops you from pretending it isn't there.

In [ ]:
def to_rgb(ds_slice):
    r = np.nan_to_num(ds_slice["visual_r"].values)
    g = np.nan_to_num(ds_slice["visual_g"].values)
    b = np.nan_to_num(ds_slice["visual_b"].values)
    return np.dstack([r, g, b]).astype(np.uint8)


rgb_1996 = to_rgb(img.sel(time="1996-03-15"))
rgb_2023 = to_rgb(img.sel(time="2023-03-15"))

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
axes[0].imshow(rgb_1996)
axes[0].set_title("~1996 (Landsat 5, 133 pooled scenes)")
axes[1].imshow(rgb_2023)
axes[1].set_title("~2023 (Sentinel-2, 156 pooled scenes)")
for ax in axes:
    ax.axis("off")
fig.suptitle("Negros, cloud-free composite true colour — historical vs. recent")
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Compare these to the 2018/2024 pair from Part II — no cloud streaks, no dark data-gap patches, consistent colour balance across the whole scene. This is what the extra engineering (pooling scenes, masking clouds, taking the median) buys you.

In [ ]:
# negros_imagery.nc stores physical reflectance (0-1 float), not the raw digital numbers (~0-12000)
# negros_sentinel2.nc used -- classify_landcover_2D's brightness<800 threshold doesn't transfer as-is.
# Re-picked by eye against this file's own numbers, same idea, just a different scale.
def classify_landcover_composite(red, nir, brightness_thr=0.08):
    ndvi = compute_ndvi(red, nir).clip(-1, 1)   # a few 1996 pixels have near-zero red+nir -> guard the ratio
    brightness = (red + nir) / 2

    landcover = xr.full_like(red, "Bare/Urban", dtype="<U10")
    landcover.values[(brightness < brightness_thr).values & (ndvi < 0.1).values] = "Water"
    landcover.values[(ndvi > 0.3).values] = "Vegetation"
    landcover.values[np.isnan(red.values)] = ""
    return landcover


landcover_1996 = classify_landcover_composite(img["red"].sel(time="1996-03-15"), img["nir"].sel(time="1996-03-15"))
landcover_2023 = classify_landcover_composite(img["red"].sel(time="2023-03-15"), img["nir"].sel(time="2023-03-15"))

print("1996:", dict(zip(*np.unique(landcover_1996.values, return_counts=True))))
print("2023:", dict(zip(*np.unique(landcover_2023.values, return_counts=True))))

> 💬 **Discussion:** Look at the **Water** counts: they jump from under 1% of the scene in 1996 to over 40% in 2023, using the exact same rule. Negros' coastline didn't move — this is Landsat 5 and Sentinel-2 simply not being perfectly cross-calibrated in absolute brightness, a real, known challenge when mixing sensors across decades. We won't chase that down here; it means the Water/Bare-Urban split isn't trustworthy across these two dates, so we'll focus on **Vegetation** only, which — reassuringly — barely moves if you nudge `brightness_thr` up or down (unlike Water). That robustness is exactly why NDVI-based vegetation detection is the workhorse of real remote-sensing change detection, not brightness thresholds.

In [ ]:
lost_vegetation_composite = detect_change(landcover_1996, landcover_2023, from_class="Vegetation")
frac_lost_composite = float(lost_vegetation_composite.sum()) / int((landcover_1996 == "Vegetation").sum())
print(f"Composite-based: {frac_lost_composite * 100:.1f}% of 1996's Vegetation-classified pixels changed by 2023")

# Fairer Hansen benchmark: full loss record (2001-2023) against the 2000 forest baseline,
# instead of just the 2018-2023 window used in subsection 2
lost_full_record = (lossyear >= 1) & (lossyear <= 23)
frac_lost_hansen_full = float((lost_full_record & had_forest_2000).sum()) / float(had_forest_2000.sum())
print(f"Hansen, full 2001-2023 record: {frac_lost_hansen_full * 100:.1f}% of the 2000 forest baseline lost")

print(f"\nFor comparison -- subsection 2's numbers over the flawed 2018-2024 pair:")
print(f"  Our classifier: {frac_lost_ours * 100:.1f}%   Hansen (2018-2023 only): {frac_lost * 100:.1f}%")

> 💬 **Discussion:** That's a night-and-day difference from subsection 2. With proper season-matched, cloud-masked composites, our homemade classifier's vegetation-loss estimate (**~3%**) lines up closely with Hansen's own independent estimate over a comparable era (**~3.5%**) — instead of being off by more than **15x** (18% vs. ~1%) the way the raw 2018/2024 snapshots were. Same classifier, same three-line NDVI rule, same `detect_change` function — the only thing that changed was the *input data's* quality. That's the actual lesson of Day 2's satellite imagery work: a simple model on well-engineered data beats a simple model on convenient-but-flawed data, every time. It's still not perfect — this is a longer, differently-sensored window than Hansen's, and we already flagged that Water/Bare-Urban isn't reliable here — but the number you'd actually act on is far more trustworthy now.

---

### 4) Urban expansion

Same `detect_change` function, different arguments:

In [ ]:
gained_urban = detect_change(landcover_2018, landcover_2024, to_class="Bare/Urban")    # ✏️ to_class="Bare/Urban", from_class left as default (any)
print(f"Pixels that newly became Bare/Urban by 2024: {int(gained_urban.sum())}")

Zoom in on Bacolod itself:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.pcolormesh(gained_urban.longitude, gained_urban.latitude, gained_urban,
              cmap=mcolors.ListedColormap(["#eeeeee", "saddlebrown"]))
ax.scatter(122.95, 10.68, color="black", s=40, zorder=5)
ax.text(123.0, 10.68, "Bacolod", fontsize=9)

ax.set_xlim(122.75, 123.15)           # ✏️ zoom to the Bacolod area
ax.set_ylim(10.5, 10.9)
ax.set_title("Newly urban/bare pixels near Bacolod, 2018 -> 2024")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

Quick summary numbers:

In [ ]:
urban_2018 = float((landcover_2018 == "Bare/Urban").mean())      # ✏️ (landcover_2018 == "Bare/Urban").mean()
urban_2024 = float((landcover_2024 == "Bare/Urban").mean())      # ✏️ same, for landcover_2024

print(f"Bare/Urban fraction of Negros — 2018: {urban_2018 * 100:.1f}%,  2024: {urban_2024 * 100:.1f}%")

> 💬 **Discussion:** Bare/Urban area nearly doubled (≈12% vs ≈7%) between 2018 and 2024 in our own classifier. Does the map show visible urban growth around Bacolod? *(You may notice the area immediately around the marker itself looks unchanged — that's the 2018 cloud-gap we already flagged, not an absence of growth; look at the clearly-changed patches further east/north instead.)* Keep in mind our "Bare/Urban" class can't distinguish a new subdivision from a newly-cleared or fallow field — what other data would help you separate those two stories?

---

> 💬 **Final discussion:**
> - Look back at Part II's Philippine Eagle map and the EN/CR/VU mammal species you found on Negros. Does the forest-loss and urban-expansion pattern you just mapped help explain why those species are threatened?
> - Across all of Day 2, you built **two** homemade classifiers from raw data — one for climate (Part I), one for land cover (Part III) — validated both against independent "official" products (Beck et al., Hansen), and used the second one to directly measure real-world change over time. That loop (rule -> function -> grid -> map -> validate) is the core pattern behind an enormous amount of real environmental data science.

---

## Well done — Day 2 complete!

Across today you:
- Learned `def` functions and `for` loops (including nested loops), and used them to build a full climate classification pipeline from scratch
- Classified the entire Philippines at three points in time, and validated your homemade model against Beck et al. (2023)
- Explored forest cover, satellite imagery, and real (messy!) biodiversity data for Negros
- Built a second homemade classifier — this time for land cover — reusing the exact same rule -> function -> grid -> map pattern from Part I
- Measured real land-use change on Negros using your own classifier, cross-checked against Hansen et al.'s independent forest-loss product
- Connected climate, vegetation, satellite imagery, biodiversity, and land-use change into a single, coherent story about one island

Tomorrow: Day 3.